# Sentimento de notícias e retornos de cinco ações dos EUA (2017–2019)

**Caio Ladislau do Couto · Alice Barros Lourencini Palaoro · Mateus Henrique da Costa Resende**  

FGV EMAp · Análise Exploratória de Dados e Visualização · 2026.2 · Professor João Rulff

**Entrega intermediária de 29/10/2026 · preparação inicial em 16/10/2026 · revisão didática em 30/10/2026**  

**Repositório:** [caiocouto30-boop/projeto-sentimento-acoes](https://github.com/caiocouto30-boop/projeto-sentimento-acoes)

## Resumo

Investigamos associações entre o tom de manchetes e os retornos de NVDA, NFLX, BAC, JNJ e KO em 2017–2019. Combinamos notícias do Kaggle, cotações do Yahoo Finance e o modelo pronto FinBERT. A limpeza manteve 6.552 associações manchete–empresa; 2.387 pares empresa/pregão têm notícias e janela completa para as comparações. A cobertura é desigual, as distribuições de retorno por classe se sobrepõem e as correlações são pequenas. Uma regra textual conservadora identifica 621 associações com relatos explícitos de preço já observado. Após sua exclusão e a reagregação, restam 2.271 pares; ao exigir também três notícias, 697. As correlações variam entre empresas e filtros, sem relação uniforme ou robustez geral. Os sete conjuntos de resultados são descritivos; a marcação automática de retrospectivas e os vínculos de ticker são limitações, sem identificação causal.

**Palavras-chave:** análise exploratória; visualização; notícias financeiras; sentimento; retornos.

**Versão para a entrega final — revisão de 15/11/2026.** O notebook é o documento principal; o HTML é sua exportação para leitura. Esta revisão preserva a sequência e os gráficos Altair do trabalho original e acrescenta a comparação de duas definições de intensidade.

## 1. Introdução

Manchetes financeiras podem acompanhar mudanças nas expectativas ou relatar movimentos já ocorridos. Comparar seu tom com os retornos exige conhecer a cobertura da fonte, medir sentimento e alinhar corretamente notícias e pregões.

Estudamos NVIDIA (NVDA), Netflix (NFLX), Bank of America (BAC), Johnson & Johnson (JNJ) e Coca-Cola (KO), em 2017–2019. O recorte foi escolhido pela disponibilidade de notícias nos três anos e não representa todo o mercado americano. O FinBERT serve à preparação; a contribuição central é a análise visual.

### Hipóteses exploratórias

**H1:** a cobertura varia entre empresas e períodos. **H2:** maior intensidade de sentimento acompanha maiores retornos absolutos posteriores. **H3:** a associação entre sentimento e retorno varia entre empresas e anos. Essas hipóteses organizam a investigação, sem alegação de pré-registro ou exigência de confirmação. As sete perguntas progridem da cobertura à robustez.

Na revisão final, H2 é examinada também pela média dos módulos dos escores individuais. Essa comparação é uma análise de sensibilidade posterior à definição original, não uma nova hipótese pré-registrada.

## 2. Dados e unidades de observação

As notícias vêm de *Daily Financial News for 6000+ Stocks*, de Miguel Aenlle, no Kaggle [1]. O arquivo contém manchete, data, ticker, URL e `publisher` (autor ou fonte conforme o registro). Fechamento ajustado e volume vêm do Yahoo Finance via `yfinance` [2].

| Representação | Unidade | Tamanho |
| --- | --- | --- |
| Arquivo original | Registro associado a uma ação | 1.407.328 |
| Recorte antes/depois da limpeza | Empresa/data civil/manchete | 6.556 / 6.552 |
| Textos classificados | Manchete distinta | 6.019 |
| Agregado por data civil | Empresa/data da notícia | 2.569 |
| Cotações com margem | Empresa/pregão | 772 por empresa |
| Painel 2017–2019 | Empresa/pregão de resposta | 3.770 |

As perguntas 1–2 usam associações manchete–empresa; a 3 usa empresa/data civil; as 4–7 usam empresa/pregão. Textos compartilhados não são observações independentes. Ausência no corpus não significa ausência de notícias no mercado.


### Como executar e explorar

Selecione o ambiente Python com as dependências de `requirements-notebook.txt`, execute as células em ordem e salve o notebook. Os dados preparados estão em `data/processed/`. A preparação completa permanece em `01_preparacao_dados.ipynb`.

As transformações, agregações e construções dos gráficos estão nas células deste notebook. As funções da regra de retrospectivas continuam em `tools/robustez.py`, como na versão original. Os controles filtram as visualizações; as tabelas e a narrativa descrevem a amostra completa indicada. O GitHub pode não executar os gráficos; abra o `.ipynb` no Jupyter ou exporte o HTML depois da execução.

No terminal, `python tools/executar_artigo.py` executa o notebook em um kernel Jupyter. Depois, `python tools/exportar_artigo.py` exporta o notebook salvo sem reconstruir suas células.

In [1]:
from pathlib import Path
import os
os.environ.setdefault('MKL_THREADING_LAYER', 'SEQUENTIAL')  # Compatibilidade do ambiente local.
import pandas as pd
import altair as alt
from IPython.display import display

RAIZ = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA = RAIZ / 'data/processed'
noticias = pd.read_csv(DATA / 'noticias_com_sentimento.csv', parse_dates=['news_date'])
painel = pd.read_csv(DATA / 'base_integrada_pregoes.csv', parse_dates=['trade_date'])
campos_analise = ['stock', 'trade_date', 'n_noticias', 'escore_sentimento',
                  'intensidade_sentimento', 'classe_sentimento', 'retorno_pct', 'retorno_absoluto_pct']
principal = painel.loc[painel['tem_noticias'] & painel['elegivel_analise_principal'], campos_analise].copy()
principal['ano'] = principal['trade_date'].dt.year

# Conferir as unidades e preservar ausência de sentimento sem notícias.
assert not noticias.duplicated(['stock', 'news_date', 'headline']).any()
assert not painel.duplicated(['stock', 'trade_date']).any()
assert painel.loc[~painel['tem_noticias'], 'escore_sentimento'].isna().all()
assert principal[['escore_sentimento', 'retorno_pct']].notna().all().all()

EMPRESAS = ['BAC', 'JNJ', 'KO', 'NFLX', 'NVDA']
CLASSES = ['negativo', 'neutro', 'positivo', 'empate']
CORES = ['#BD5B3A', '#83929F', '#16877B', '#8664A5']
ESCALA_CLASSES = alt.Scale(domain=CLASSES, range=CORES)
alt.theme.enable('default')
# HTML no Jupyter e especificação JSON válida para exportação.
alt.renderers.enable('html')
render_html = alt.renderers.get()
def render_artigo(spec):
    bundle = render_html(spec)
    if 'text/html' in bundle:
        estilo = '<style>#vg-tooltip-element td.value{white-space:pre-wrap;overflow-wrap:anywhere;max-width:620px;text-align:left}#vg-tooltip-element{font:13px/1.5 Arial,sans-serif;max-width:720px}</style>'
        bundle['text/html'] = estilo + bundle['text/html']
    bundle['application/vnd.vegalite.v6+json'] = spec
    return bundle

alt.renderers.register('artigo', render_artigo)
alt.renderers.enable('artigo')

def spearman(df, x, y):
    pares = df[[x, y]].dropna()
    if len(pares) < 3 or (pares.nunique() < 2).any():
        return float('nan')
    return pares[x].rank().corr(pares[y].rank())

print(f'{len(noticias):,} associações; {len(painel):,} pares no painel; '
      f'{len(principal):,} pares na amostra principal.')

6,552 associações; 3,770 pares no painel; 2,387 pares na amostra principal.


## 3. Método

### Limpeza e sentimento

Selecionamos as cinco empresas e 2017–2019, preservando valores e identificadores originais. Normalizamos somente espaços nas manchetes e retiramos quatro duplicatas por empresa/data/manchete. Textos compartilhados entre empresas foram sinalizados e mantidos. Preços e volumes foram validados, sem preenchimento artificial. A limpeza, o FinBERT e a integração estão no [notebook de preparação](notebooks/01_preparacao_dados.ipynb).

Usamos `ProsusAI/finbert`, revisão `4556d13015211d73dccd3fdd39d39232506f3e43`, sem treinamento adicional [3–4]. O escore de cada manchete é:

$$s_i=p_i(\mathrm{positivo})-p_i(\mathrm{negativo}).$$

A classe agregada é a de maior probabilidade média, com empate explícito à tolerância de $10^{-8}$. O escore agregado é a média dos escores, e a intensidade é seu módulo. Tons opostos podem se compensar; escore próximo de zero não define neutralidade. Probabilidades softmax não são estimativas calibradas de acerto.

A revisão dirigida de 50 manchetes por `caio` registrou 39 concordâncias e 11 divergências. Um único revisor, a seleção de casos e a exposição às previsões impedem interpretar 39/50 como acurácia geral. Os rótulos humanos não substituíram as classes automáticas ([relatório](docs/finbert.md)).

### Alinhamento e retornos

Os horários não permitem ordenação intradiária confiável. Preservamos a data civil e associamos cada notícia de data $d$ ao primeiro pregão $t>d$. O snapshot de preços de 09/10/2026 cobre 15/12/2016–10/01/2020 e foi comparado ao calendário XNYS [5]. Depois do mapeamento, reagrupamos por empresa/pregão com peso igual por manchete, evitando médias de dias com quantidades diferentes.

$$r_{e,t}=100\left(\frac{P^{aj}_{e,t}}{P^{aj}_{e,t^-}}-1\right),\qquad a_{e,t}=|r_{e,t}|.$$

$t^-$ é o pregão anterior. Calculamos o retorno na série completa antes do recorte. Retorno absoluto mede oscilação diária, não volatilidade formal. Sem notícias, somente a contagem é zero; escore, probabilidades e classe permanecem ausentes.

O painel tem 3.770 pares, dos quais 2.390 têm notícias. Três desses pares têm janela inicial incompleta, deixando **2.387** para a comparação principal. Seis notícias de 31/12/2019 apontam para janeiro de 2020 e permanecem como margem fora do painel. As bordas estão documentadas na [integração](docs/integracao.md).

### Duas definições de intensidade

Preservamos a definição original, agora chamada **intensidade líquida**, e acrescentamos uma sensibilidade:

$$I_{\mathrm{líquida}}=\left|\frac{1}{n}\sum_i s_i\right|,\qquad I_{\mathrm{média}}=\frac{1}{n}\sum_i|s_i|.$$

A segunda evita a compensação entre escores de sinais opostos. Ambas dependem do FinBERT e não esgotam o conceito de intensidade emocional de uma notícia. As fórmulas são calculadas nas células da Q5 e reagregadas após as exclusões da Q7.

### Exemplo: da manchete ao retorno

O registro `source_id = 899818` anuncia **“Netflix Announces 'Love Alarm' as First Korean Original Series”**, em 05/01/2017. O percurso nas bases preparadas é:

| Etapa | Valor observado |
| --- | --- |
| Probabilidades da manchete | Positivo: 0,1811; negativo: 0,0160; neutro: 0,8029 |
| Classe e escore individuais | Classe neutra (maior probabilidade); escore = P(positivo) − P(negativo) ≈ +0,1651 |
| Associação temporal | Notícia de 05/01/2017 → primeiro pregão posterior, 06/01/2017 |
| Agregação da janela | Duas manchetes para NFLX/pregão; escore médio ≈ +0,3335 e classe agregada positiva |
| Fechamentos ajustados | Anterior ≈ 13,1810; atual ≈ 13,1070 |
| Retorno do pregão | 100 × (preço atual / preço anterior − 1) ≈ −0,5614% |

Os cálculos usam os valores completos; a tabela os arredonda. A outra manchete da janela tem escore ≈ +0,5018, e a média dos dois escores é ≈ +0,3335. A classe do grupo resulta da maior probabilidade média, não de uma votação das classes individuais. Há **um único retorno** para o par NFLX/pregão. Assim, classe, escore e retorno são medidas diferentes; os valores não demonstram que a manchete causou a queda.


## 4. Pergunta 1 — Como a cobertura de notícias se distribui entre as cinco empresas e ao longo do período?

**Transformação.** Contamos associações por empresa/ano e empresa/mês. Reindexamos os 36 meses para mostrar zeros de cobertura, sem atribuir sentimento a dias sem notícias. A tabela apresenta os totais anuais; o mapa e as barras usam a mesma base.

**Interação:** clique em uma empresa para destacá-la nos dois painéis; dê duplo clique no fundo para limpar. O cursor mostra a contagem exata. Fonte: notícias limpas do Kaggle, n = 6.552 associações.


In [2]:
cobertura = (noticias.assign(ano=noticias['news_date'].dt.year)
             .groupby(['stock', 'ano']).size().unstack(fill_value=0))
display(cobertura.assign(Total=cobertura.sum(axis=1)).rename_axis('Empresa'))

indice_mensal = pd.MultiIndex.from_product(
    [EMPRESAS, pd.date_range('2017-01-01', '2019-12-01', freq='MS')],
    names=['stock', 'mes'])
mensal = (noticias.assign(mes=noticias['news_date'].dt.to_period('M').dt.to_timestamp())
          .groupby(['stock', 'mes']).size().reindex(indice_mensal, fill_value=0)
          .rename('n').reset_index())
totais = noticias.groupby('stock').size().rename('n').reset_index()
destaque = alt.selection_point(fields=['stock'], name='empresa_cobertura')
mapa = alt.Chart(mensal).mark_rect().encode(
    x=alt.X('mes:O', title='Mês da notícia', timeUnit='yearmonth',
            axis=alt.Axis(labelExpr="timeFormat(datum.value, '%b %y')", labelAngle=-55)),
    y=alt.Y('stock:N', title='Empresa', sort=EMPRESAS),
    color=alt.Color('n:Q', title='Notícias', scale=alt.Scale(scheme='blues', zero=True)),
    opacity=alt.condition(destaque, alt.value(1), alt.value(.2)),
    tooltip=['stock:N', alt.Tooltip('mes:T', title='Mês', format='%Y-%m'), 'n:Q']
).properties(width=610, height=200).add_params(destaque)
barras = alt.Chart(totais).mark_bar(color='#32648D').encode(
    x=alt.X('n:Q', title='Total', scale=alt.Scale(zero=True)),
    y=alt.Y('stock:N', title=None, sort=EMPRESAS, axis=None),
    opacity=alt.condition(destaque, alt.value(1), alt.value(.2)),
    tooltip=['stock:N', alt.Tooltip('n:Q', title='Associações')]
).properties(width=150, height=200)
q1 = (mapa | barras).properties(title='1 · A cobertura difere entre empresas e meses')
display(q1)


ano,2017,2018,2019,Total
Empresa,,,,
BAC,348,220,303,871
JNJ,300,314,432,1046
KO,238,195,211,644
NFLX,759,784,801,2344
NVDA,614,526,507,1647


alt.HConcatChart(...)

**Resultado.** NFLX tem a maior cobertura nos três anos (2.344 associações) e KO a menor (644). NVDA reúne 1.647, JNJ 1.046 e BAC 871. O mapa mostra variação mensal, e a tabela anual confirma trajetórias diferentes entre empresas. H1 encontra apoio descritivo: as comparações posteriores partem de quantidades desiguais de informação.

**Limitação.** Contagens descrevem a cobertura do corpus, sem explicar suas causas ou representar todas as notícias do mercado. Uma manchete pode contribuir para várias empresas.


## 5. Pergunta 2 — Quais empresas apresentam maiores proporções de manchetes positivas, neutras e negativas?

**Transformação.** Contamos classes por empresa e dividimos pelo total da própria empresa. O denominador e as contagens aparecem na tabela e no cursor; as barras somam 100%. A unidade é a associação manchete–empresa, antes do filtro de elegibilidade de retornos.

**Interação:** passe o cursor sobre um segmento para consultar classe, percentual, quantidade e total. Fonte: Kaggle e FinBERT, n = 6.552.


In [3]:
classes = pd.crosstab(noticias['stock'], noticias['classe_finbert']).reindex(
    index=EMPRESAS, columns=CLASSES, fill_value=0)
display(classes.assign(total=classes.sum(axis=1)))
composicao = classes.rename_axis(columns='classe').stack().rename('n').reset_index()
composicao['total'] = composicao.groupby('stock')['n'].transform('sum')
composicao['proporcao'] = composicao['n'] / composicao['total']
composicao['ordem'] = composicao['classe'].map({c: i for i, c in enumerate(CLASSES)})
q2 = alt.Chart(composicao).mark_bar().encode(
    x=alt.X('proporcao:Q', stack='zero', title='Proporção dentro da empresa',
            axis=alt.Axis(format='.0%'), scale=alt.Scale(domain=[0, 1])),
    y=alt.Y('stock:N', title='Empresa', sort=EMPRESAS),
    color=alt.Color('classe:N', title='Sentimento', scale=ESCALA_CLASSES),
    order='ordem:Q',
    tooltip=['stock:N', 'classe:N', alt.Tooltip('proporcao:Q', format='.1%'), 'n:Q', 'total:Q']
).properties(width=760, height=220, title='2 · Neutro é a classe mais frequente')
display(q2)


classe_finbert,negativo,neutro,positivo,empate,total
stock,,,,,
BAC,185,524,162,0,871
JNJ,224,516,306,0,1046
KO,108,398,138,0,644
NFLX,429,1359,556,0,2344
NVDA,395,801,451,0,1647


alt.Chart(...)

**Resultado.** Neutro é a classe mais frequente nas cinco empresas, sem superar 50% em JNJ e NVDA. KO tem a maior proporção neutra (61,8%), JNJ a maior positiva (29,3%) e NVDA a maior negativa (24,0%). As proporções permitem comparar composição sem confundi-la com cobertura.

**Limitação.** O modelo classifica o texto inteiro e pode atribuir o mesmo tom a tickers com papéis diferentes na manchete. A revisão humana dirigida não estima seu erro geral.


## 6. Pergunta 3 — Como o sentimento agregado varia ao longo do tempo e quais períodos concentram extremos?

**Transformação e leitura.** Recalculamos a média do escore e a contagem por empresa/data civil. O tamanho dos pontos indica a contagem; o contorno escuro destaca mínimo e máximo, com desempate pela data mais antiga. A escala permanece em −1 a 1, sem preencher dias sem notícias.

**Interação:** escolha a empresa e arraste um intervalo no painel inferior para ampliar as datas no superior. Duplo clique limpa o intervalo. O cursor mostra valores e uma manchete de maior |escore individual| da data, identificada como exemplo, não como síntese do grupo. A tabela mantém os dez extremos de todas as empresas. Fonte: Kaggle e FinBERT, n = 2.569 grupos.

**Imagem estática:** a seleção inicial é BAC; no HTML, use o seletor para mudar a empresa.


### Etapa 1 — Preparar os grupos e localizar os extremos

Reunimos as manchetes por empresa/data civil e identificamos mínimo e máximo de cada empresa.


In [4]:
diario = noticias.groupby(['stock', 'news_date'], as_index=False).agg(
    escore_sentimento=('escore_sentimento', 'mean'), n_noticias=('headline', 'size'))
exemplos = (noticias.assign(intensidade=noticias['escore_sentimento'].abs())
            .sort_values(['intensidade', 'source_row'], ascending=[False, True])
            .drop_duplicates(['stock', 'news_date'])
            [['stock', 'news_date', 'headline']].rename(columns={'headline': 'manchete_exemplo'}))
diario = diario.merge(exemplos, on=['stock', 'news_date'], validate='one_to_one')
extremos = pd.concat([
    diario.sort_values(['escore_sentimento', 'news_date']).groupby('stock').head(1).assign(extremo='mínimo'),
    diario.sort_values(['escore_sentimento', 'news_date'], ascending=[False, True])
          .groupby('stock').head(1).assign(extremo='máximo')])
diario = diario.merge(extremos[['stock', 'news_date', 'extremo']],
                      on=['stock', 'news_date'], how='left', validate='one_to_one')

# Até três manchetes de exemplo por grupo, ordenadas pelo módulo do escore.
# A primeira conserva o critério da manchete ilustrativa original.
titulos = (noticias.assign(modulo=noticias['escore_sentimento'].abs())
    .sort_values(['modulo', 'source_row'], ascending=[False, True])
    .groupby(['stock', 'news_date'])['headline'].agg(lambda s: s.head(3).tolist())
    .rename('titulos').reset_index())
for i in range(3):
    titulos[f'manchete_{i+1}'] = titulos['titulos'].map(lambda xs: xs[i] if len(xs) > i else '—')
diario = diario.merge(titulos.drop(columns='titulos'), on=['stock', 'news_date'], validate='one_to_one')
diario['outras_manchetes'] = (diario['n_noticias'] - 3).clip(lower=0)

### Etapa 2 — Explorar a série temporal

O gráfico começa em BAC. O seletor muda a empresa; a seleção de datas altera o intervalo mostrado.

**Retorno do professor incorporado:** o tooltip apresenta empresa, data, escore médio, quantidade e até três manchetes completas. Quando há mais notícias, o campo “Outras manchetes” informa a quantidade adicional. Os títulos são exemplos ordenados por módulo do escore individual; não representam sozinhos o sentimento médio do ponto.

In [5]:
empresa_tempo = alt.param(name='empresa_tempo', value='BAC',
    bind=alt.binding_select(options=EMPRESAS, name='Empresa: '))
intervalo = alt.selection_interval(encodings=['x'], name='intervalo_datas')
pontos = alt.Chart(diario).transform_filter(alt.datum.stock == empresa_tempo).mark_circle().encode(
    x=alt.X('news_date:T', title='Data civil da notícia'),
    y=alt.Y('escore_sentimento:Q', title='Escore médio', scale=alt.Scale(domain=[-1, 1])),
    size=alt.Size('n_noticias:Q', title='Notícias', scale=alt.Scale(range=[18, 180])),
    color=alt.Color('escore_sentimento:Q', title='Escore',
        scale=alt.Scale(domain=[-1, 0, 1], range=['#BD5B3A', '#E3E8EC', '#16877B'])),
    stroke=alt.condition('isValid(datum.extremo)', alt.value('#172533'), alt.value('transparent')),
    strokeWidth=alt.value(2), opacity=alt.value(.8),
    tooltip=[alt.Tooltip('stock:N', title='Empresa'),
             alt.Tooltip('news_date:T', title='Data da notícia', format='%d/%m/%Y'),
             alt.Tooltip('escore_sentimento:Q', title='Escore médio', format='.3f'),
             alt.Tooltip('n_noticias:Q', title='Notícias no grupo'),
             alt.Tooltip('manchete_1:N', title='Manchete 1 (maior |escore|)'),
             alt.Tooltip('manchete_2:N', title='Manchete 2'),
             alt.Tooltip('manchete_3:N', title='Manchete 3'),
             alt.Tooltip('outras_manchetes:Q', title='Outras manchetes no grupo')])
detalhe = pontos.transform_filter(intervalo).properties(width=760, height=270)
visao_geral = pontos.encode(size=alt.value(12)).properties(
    width=760, height=70, title='Arraste aqui para selecionar datas').add_params(intervalo)
q3 = (detalhe & visao_geral).add_params(empresa_tempo).properties(
    title=alt.TitleParams(
        text='3 · Extremos diários têm pouco suporte de manchetes',
        subtitle='Imagem inicial: BAC; use o seletor para mudar a empresa.'))
display(q3)
display(extremos.sort_values(['stock', 'extremo'])[
    ['stock', 'extremo', 'news_date', 'escore_sentimento', 'n_noticias', 'manchete_exemplo']])

alt.VConcatChart(...)

,stock,extremo,news_date,escore_sentimento,n_noticias,manchete_exemplo
304,BAC,máximo,2019-01-30,0.934606,1,PACCAR's Forecast Sees A Slightly Growth In Cl...
381,BAC,mínimo,2019-08-09,-0.963682,1,Shares of several blue-chip stocks are trading...
720,JNJ,máximo,2018-10-24,0.938982,1,Janssen's Stelara Demonstrated Sustained Effic...
612,JNJ,mínimo,2018-02-05,-0.961099,1,"J&J Shares Fall 3.5%, Quickly Rebound From Low..."
1197,KO,máximo,2019-04-03,0.940817,1,Guggenheim Referencing Nielsen Data Notes Coca...
1246,KO,mínimo,2019-10-02,-0.966257,1,Coca-Cola Co shares are trading lower in sympa...
1851,NFLX,máximo,2019-06-17,0.932194,1,Netflix shares are trading higher on reports o...
1609,NFLX,mínimo,2018-05-10,-0.962488,1,"Netflix Shares Down ~0.7% Over Last Few Mins.,..."
2515,NVDA,máximo,2019-09-25,0.933611,1,NVIDIA shares are trading higher despite no co...
2409,NVDA,mínimo,2019-03-06,-0.965972,1,Semiconductor Industry Association Reports 7.2...


### Etapa 3 — Repetir com pelo menos três manchetes

Recalculamos apenas a escolha dos extremos nas datas com maior suporte, preservando a seleção principal.


In [6]:
diario_apoiado = diario.loc[diario['n_noticias'].ge(3)].copy()
extremos_suporte = pd.concat([
    diario_apoiado.sort_values(['escore_sentimento', 'news_date']).groupby('stock').head(1).assign(extremo='mínimo'),
    diario_apoiado.sort_values(['escore_sentimento', 'news_date'], ascending=[False, True])
        .groupby('stock').head(1).assign(extremo='máximo')])
display(extremos_suporte[['stock', 'extremo', 'news_date', 'escore_sentimento', 'n_noticias']].round({'escore_sentimento': 3}))


,stock,extremo,news_date,escore_sentimento,n_noticias
285,BAC,mínimo,2018-12-04,-0.950,3
2243,NVDA,mínimo,2018-04-02,-0.928,3
557,JNJ,mínimo,2017-09-20,-0.815,3
1478,NFLX,mínimo,2017-10-18,-0.675,3
1191,KO,mínimo,2019-03-12,-0.554,5
1254,KO,máximo,2019-10-23,0.881,3
2460,NVDA,máximo,2019-06-11,0.877,3
833,JNJ,máximo,2019-07-10,0.818,3
334,BAC,máximo,2019-04-12,0.797,3
1764,NFLX,máximo,2019-01-10,0.640,3


**Resultado.** Os dez extremos sem restrição têm uma manchete cada. Em BAC, o máximo menciona PACCAR, mostrando um vínculo indireto de ticker. Para verificar o suporte, repetimos a seleção apenas em datas com três ou mais notícias. Os extremos ficam menos próximos de ±1: em NFLX, vão de −0,675 a +0,640; em KO, de −0,554 a +0,881. O mínimo de BAC em 04/12/2018 (−0,950; três manchetes) reúne textos sobre queda do setor bancário. O máximo de JNJ em 10/07/2019 (+0,818; três) reúne decisões sobre o litígio do talco.

**Limitação.** Mais manchetes não garantem relevância para o ticker ou fontes independentes. O máximo apoiado de NFLX inclui um título sobre o setor de cannabis; o mínimo de KO inclui uma manchete sobre Monster. Conservamos os registros e exportamos todos os textos dos extremos para permitir a inspeção. O sentimento do texto inteiro pode refletir outras empresas e notícias retrospectivas.


## 7. Pergunta 4 — Como se distribuem os retornos do pregão seguinte a dias com sentimento positivo, neutro ou negativo?

**Transformação.** Selecionamos os 2.387 pares com notícias e janela completa, sem repetir retornos por manchete. Calculamos tamanho, mediana e quartis por empresa/classe. As caixas usam Q1, mediana e Q3 e bigodes de 1,5 IQR; os extremos continuam visíveis e a escala de retorno é comum.

**Interação:** o cursor consulta os resumos das caixas e os retornos extremos. As contagens por classe ficam na tabela calculada abaixo. Fonte: painel integrado; retorno em porcentagem.


In [7]:
def resumir_classes(df):
    return df.groupby(['stock', 'classe_sentimento'])['retorno_pct'].agg(
        n='size', media='mean', mediana='median', minimo='min', maximo='max',
        q1=lambda s: s.quantile(.25), q3=lambda s: s.quantile(.75)).reset_index()

retornos_classes = resumir_classes(principal)
display(retornos_classes.round(3))
q4 = alt.Chart(principal[['stock', 'classe_sentimento', 'retorno_pct']]).mark_boxplot(extent=1.5, size=36).encode(
    x=alt.X('classe_sentimento:N', title='Classe agregada', sort=CLASSES,
            axis=alt.Axis(labelAngle=0)),
    y=alt.Y('retorno_pct:Q', title='Retorno do pregão (%)'),
    color=alt.Color('classe_sentimento:N', scale=ESCALA_CLASSES, legend=None),
    tooltip=['stock:N', 'classe_sentimento:N', 'retorno_pct:Q']
).properties(width=235, height=230).facet(
    facet=alt.Facet('stock:N', title=None, sort=EMPRESAS), columns=3
).properties(title='4 · As distribuições por classe se sobrepõem')
display(q4)


,stock,classe_sentimento,n,media,mediana,minimo,maximo,q1,q3
0,BAC,negativo,88,-0.138,0.000,-5.810,4.354,-0.731,0.621
1,BAC,neutro,289,0.129,0.000,-5.919,7.156,-0.565,0.938
2,BAC,positivo,61,-0.038,-0.061,-5.290,2.800,-1.078,1.156
3,JNJ,negativo,70,-0.244,0.092,-10.038,1.586,-0.634,0.517
4,JNJ,neutro,245,0.028,0.000,-4.259,3.545,-0.435,0.600
5,JNJ,positivo,131,0.142,0.163,-3.850,2.340,-0.380,0.614
6,KO,negativo,36,-0.027,0.095,-3.384,2.518,-0.595,0.468
7,KO,neutro,251,0.001,0.090,-8.435,2.899,-0.400,0.470
8,KO,positivo,47,0.057,0.087,-1.486,1.805,-0.444,0.634
9,NFLX,negativo,78,-0.485,-0.498,-6.271,8.462,-1.776,0.835


alt.FacetChart(...)

**Resultado.** As distribuições se sobrepõem, sem ordem de medianas por classe comum às cinco empresas. Em NFLX, a mediana é −0,498% na classe negativa e +0,015% na positiva; em NVDA, +0,382% e −0,173%, respectivamente. O rótulo não separa retornos de forma uniforme.

**Limitação.** Grupos desiguais, extremos, retrospectivas e ausência de horários restringem a leitura. Diferenças descritivas não demonstram significância estatística, previsibilidade ou efeito causal.


## 8. Pergunta 5 — Dias com mais notícias ou sentimento mais intenso são seguidos por maiores oscilações, medidas pelo retorno absoluto?

**Transformação.** Comparamos quantidade de notícias e módulo do escore agregado com o retorno absoluto nos mesmos 2.387 pares. Spearman é a correlação dos postos médios, usada apenas como resumo descritivo. As duas dispersões usam todos os pontos e a mesma escala vertical, sem corte de extremos.

**Interação:** escolha a empresa e arraste uma seleção no painel esquerdo para destacar os mesmos pregões no direito. Duplo clique limpa. O cursor mostra empresa, data, n, retorno e uma manchete de maior |escore individual| da janela. Essa manchete é um exemplo, não explica isoladamente o escore agregado. A tabela de coeficientes refere-se à amostra completa de cada empresa. Fonte: painel integrado.

Na revisão final, os mesmos 2.387 pares são comparados pelas duas medidas. O seletor “Medida” altera o eixo do painel direito; o painel esquerdo e os retornos permanecem os mesmos. A tabela registra os coeficientes das duas definições.

### Comparar as duas definições na mesma amostra

O código abaixo deriva as medidas e calcula as associações antes de construir o gráfico.

In [8]:
# A unidade continua sendo um par empresa/pregão, com o mesmo retorno.
mapeadas = pd.read_csv(DATA / 'noticias_pregao_retornos.csv', parse_dates=['news_date', 'trade_date'])

def anexar_intensidades(base, textos):
    medias = (textos.assign(modulo=textos['escore_sentimento'].abs())
        .groupby(['stock', 'trade_date'], as_index=False)
        .agg(intensidade_media=('modulo', 'mean')))
    saida = base.drop(columns=['intensidade_media', 'intensidade_liquida'], errors='ignore').merge(
        medias, on=['stock', 'trade_date'], validate='one_to_one')
    saida['intensidade_liquida'] = saida['intensidade_sentimento']
    assert len(saida) == len(base)
    assert (saida['intensidade_media'] + 1e-10 >= saida['intensidade_liquida']).all()
    return saida

principal = anexar_intensidades(principal, mapeadas)
associacoes_q5 = pd.DataFrame([
    {'stock': empresa, 'n': len(g),
     'rho_quantidade_abs': spearman(g, 'n_noticias', 'retorno_absoluto_pct'),
     'rho_intensidade_liquida_abs': spearman(g, 'intensidade_liquida', 'retorno_absoluto_pct'),
     'rho_intensidade_media_abs': spearman(g, 'intensidade_media', 'retorno_absoluto_pct')}
    for empresa, g in principal.groupby('stock')])
display(associacoes_q5.round(4))

,stock,n,rho_quantidade_abs,rho_intensidade_liquida_abs,rho_intensidade_media_abs
0,BAC,438,0.0327,0.0112,0.0211
1,JNJ,446,0.0297,-0.0097,-0.0337
2,KO,334,0.1608,0.0548,0.0801
3,NFLX,627,0.0570,0.0301,0.0728
4,NVDA,542,0.1252,0.0160,0.0694


In [9]:
# Uma manchete ilustrativa por pregão; o retorno continua aparecendo uma única vez.
exemplos_pregao = (mapeadas.assign(intensidade=mapeadas['escore_sentimento'].abs())
    .sort_values(['intensidade', 'source_row'], ascending=[False, True])
    .drop_duplicates(['stock', 'trade_date'])[['stock', 'trade_date', 'headline']]
    .rename(columns={'headline': 'manchete_exemplo'}))
pontos_pregao = principal.merge(exemplos_pregao, on=['stock', 'trade_date'], validate='one_to_one')
empresa_disp = alt.param(name='empresa_dispersao', value='Todas',
    bind=alt.binding_select(options=['Todas'] + EMPRESAS, name='Empresa: '))
selecao_pontos = alt.selection_interval(name='pregoes_selecionados')
base_disp = alt.Chart(pontos_pregao).transform_filter(
    (empresa_disp == 'Todas') | (alt.datum.stock == empresa_disp)
).mark_circle(size=34).encode(
    y=alt.Y('retorno_absoluto_pct:Q', title='Retorno absoluto (%)',
            scale=alt.Scale(domain=[0, float(principal['retorno_absoluto_pct'].max()) * 1.05])),
    color=alt.Color('stock:N', title='Empresa', scale=alt.Scale(
        domain=EMPRESAS, range=['#32648D', '#D18C29', '#16877B', '#8664A5', '#BD5B3A'])),
    opacity=alt.condition(selecao_pontos, alt.value(.65), alt.value(.08)),
    tooltip=['stock:N', alt.Tooltip('trade_date:T', format='%Y-%m-%d'),
             'n_noticias:Q', alt.Tooltip('escore_sentimento:Q', format='.3f'),
             alt.Tooltip('retorno_pct:Q', format='.3f'),
             alt.Tooltip('intensidade_liquida:Q', title='Intensidade líquida', format='.3f'),
             alt.Tooltip('intensidade_media:Q', title='Intensidade média', format='.3f'),
             alt.Tooltip('manchete_exemplo:N', title='Manchete de exemplo')]
).properties(width=350, height=300)
quantidade = base_disp.encode(x=alt.X('n_noticias:Q', title='Número de notícias',
    scale=alt.Scale(domain=[0, int(principal['n_noticias'].max()) + 1]))).add_params(selecao_pontos)
medida_intensidade = alt.param(name='medida_intensidade', value='intensidade_liquida',
    bind=alt.binding_select(options=['intensidade_liquida', 'intensidade_media'],
        labels=['Intensidade líquida = |média(s)|', 'Intensidade média = média(|s|)'], name='Medida: '))
intensidade = base_disp.transform_calculate(
    intensidade_exibida='datum[medida_intensidade]'
).encode(x=alt.X('intensidade_exibida:Q', title='Intensidade (medida selecionada)',
                scale=alt.Scale(domain=[0, 1])))
q5 = (quantidade | intensidade).add_params(empresa_disp, medida_intensidade).properties(
    title='5 · Quantidade e intensidade diante da oscilação diária')
display(q5)

alt.HConcatChart(...)

**Resultado.** Quantidade e retorno absoluto têm correlações positivas de 0,030 a 0,161, com grande dispersão. Para intensidade, variam de −0,010 a +0,055, próximas de zero. **H2 não recebe apoio descritivo claro nesta definição.**

**Limitação.** O módulo do escore médio permite compensação entre manchetes opostas. Correlação próxima de zero não exclui relações não monotônicas, e maior cobertura pode acompanhar contextos turbulentos sem indicar causalidade.

**Sensibilidade da medida.** A intensidade líquida tem coeficientes entre aproximadamente −0,010 e +0,055; a média dos módulos individuais, entre −0,034 e +0,080. A conclusão sobre H2 permanece restrita: não aparece apoio descritivo uniforme nessas medidas. A associação fraca não prova a inexistência de qualquer relação entre notícias e preços.

## 9. Pergunta 6 — As associações entre sentimento e retorno apresentam direção e intensidade semelhantes entre empresas e anos?

**Transformação.** Calculamos Spearman por empresa/ano do pregão e no período completo, sempre sobre a amostra principal. Cada célula mostra coeficiente e n; a escala de cor permanece em −1 a 1. As dispersões permitem inspecionar a nuvem correspondente, com escalas comuns.

**Interação:** clique em uma célula do mapa para filtrar a dispersão por empresa/período. A seleção inicial é BAC no período completo. O cursor mostra o coeficiente e n no mapa e os dados do pregão na dispersão. Nenhum coeficiente é recalculado pela seleção visual. Fonte: painel integrado.

**Imagem estática:** a dispersão inicial mostra BAC no período completo (2017–2019).


In [10]:
correlacoes = []
for empresa, g in principal.groupby('stock'):
    periodos = [('2017–2019', g)] + [(str(ano), h) for ano, h in g.groupby('ano')]
    for periodo, h in periodos:
        correlacoes.append({'stock': empresa, 'periodo': periodo, 'n': len(h),
            'rho': spearman(h, 'escore_sentimento', 'retorno_pct')})
correlacoes = pd.DataFrame(correlacoes)
correlacoes['rotulo'] = correlacoes['rho'].map('{:+.3f}'.format) + ' · n=' + correlacoes['n'].astype(str)
grupo = alt.selection_point(fields=['stock', 'periodo'], name='grupo_correlacao',
    value=[{'stock': 'BAC', 'periodo': '2017–2019'}], toggle=False, clear=False)
mapa_corr = alt.Chart(correlacoes).encode(
    x=alt.X('periodo:O', title='Ano do pregão', sort=['2017', '2018', '2019', '2017–2019']),
    y=alt.Y('stock:N', title='Empresa', sort=EMPRESAS))
quadrados = mapa_corr.mark_rect().encode(
    color=alt.Color('rho:Q', title='Spearman (ρ)', scale=alt.Scale(domain=[-1, 1], scheme='redblue')),
    opacity=alt.condition(grupo, alt.value(1), alt.value(.3)),
    tooltip=['stock:N', 'periodo:N', 'n:Q', alt.Tooltip('rho:Q', format='.4f')]
).add_params(grupo)
rotulos = mapa_corr.mark_text(color='#172533', size=12).encode(text='rotulo:N')
mapa_completo = (quadrados + rotulos).properties(width=760, height=200)

# Duas representações de período; a seleção única exibe cada pregão uma vez.
periodos_pontos = pd.concat([
    pontos_pregao.assign(periodo=pontos_pregao['ano'].astype(str)),
    pontos_pregao.assign(periodo='2017–2019')], ignore_index=True)
dispersao_corr = alt.Chart(periodos_pontos).transform_filter(grupo).mark_circle(size=32, opacity=.5).encode(
    x=alt.X('escore_sentimento:Q', title='Escore agregado', scale=alt.Scale(domain=[-1, 1])),
    y=alt.Y('retorno_pct:Q', title='Retorno (%)', scale=alt.Scale(domain=[
        float(principal['retorno_pct'].min()) - 1, float(principal['retorno_pct'].max()) + 1])),
    color=alt.Color('stock:N', title='Empresa', scale=alt.Scale(
        domain=EMPRESAS, range=['#32648D', '#D18C29', '#16877B', '#8664A5', '#BD5B3A'])),
    tooltip=['stock:N', alt.Tooltip('trade_date:T', format='%Y-%m-%d'),
             alt.Tooltip('escore_sentimento:Q', format='.3f'),
             alt.Tooltip('retorno_pct:Q', format='.3f'), 'n_noticias:Q',
             alt.Tooltip('manchete_exemplo:N', title='Manchete de exemplo')]
).properties(width=760, height=270)
q6 = (mapa_completo & dispersao_corr).properties(
    title=alt.TitleParams(
        text='6 · Correlações pequenas e sinais variáveis entre anos',
        subtitle='Dispersão inicial: BAC · 2017–2019; clique no mapa para mudar.'))
display(q6)
display(correlacoes.drop(columns='rotulo').round(4))


alt.VConcatChart(...)

,stock,periodo,n,rho
0,BAC,2017–2019,438,0.0150
1,BAC,2017,163,0.0428
2,BAC,2018,123,-0.0259
3,BAC,2019,152,0.0033
4,JNJ,2017–2019,446,0.0307
5,JNJ,2017,144,0.0691
6,JNJ,2018,143,-0.0745
7,JNJ,2019,159,0.1226
8,KO,2017–2019,334,0.0023
9,KO,2017,133,-0.0107


**Resultado.** No período completo, ρ varia de −0,041 (NVDA) a +0,058 (NFLX). Nos grupos anuais, vai de −0,153 (NVDA/2019) a +0,123 (JNJ/2019). H3 encontra variação de sinal e magnitude, mas os coeficientes pequenos não demonstram diferenças estruturais.

**Limitação.** Não foram realizados testes de diferença entre coeficientes. Dependência temporal, manchetes compartilhadas e tamanhos desiguais impedem tratar os grupos como estimativas independentes.


## 10. Pergunta 7 — Os padrões permanecem ao excluir retrospectivas identificadas pela regra e exigir pelo menos três notícias?

**Transformação.** Comparamos quatro cenários: principal; mínimo de três; sem retrospectivas identificadas pela regra; e os dois filtros combinados. A regra textual foi fixada antes dos novos coeficientes, usando somente manchetes. Consideramos explícitos relatos de negociação, variação de cotação, máximas/mínimas já observadas e resumos intradiários. Previsões, agendas de resultados e mudanças operacionais não são suficientes. A implementação e os exemplos estão no [protocolo](docs/retrospectivas/README.md).

São 586 textos distintos marcados como retrospectivos, presentes em **621 associações**. Há ainda 297 textos com sinais não conclusivos (317 associações), mantidos nos quatro cenários principais. Dois cenários adicionais, exportados nas tabelas, retiram também esses sinais. A marcação é automática com revisão assistida; não constitui anotação humana validada.

Após excluir manchetes, recalculamos médias das probabilidades, classe, escore, intensidade e contagem por empresa/pregão. **O mínimo de três é aplicado à nova contagem.** Grupos que perdem todas as notícias deixam a comparação; não recebem sentimento neutro. As amostras são sobrepostas, e todos os retornos são preservados.

**Leitura:** cores e formas distinguem os cenários; o cursor mostra coeficiente e n. Usamos a mesma escala de −1 a 1 nas três métricas. A retenção por empresa/ano está nas tabelas exportadas.

**Alcance do filtro:** a regra pode deixar passar relatos explicitamente retrospectivos em formatos não reconhecidos. Os cenários retiram somente os textos identificados por ela; não pressupomos a remoção de todas as retrospectivas.


### Etapa 1 — Construir os cenários

A marcação seleciona quais manchetes permanecem. Cada cenário de exclusão reagrega as notícias antes de aplicar o mínimo de três.


In [11]:
import sys
sys.path.insert(0, str(RAIZ / 'tools'))
from robustez import CENARIOS, marcar_textos, construir_cenarios, resumir_retencao

marcacao = marcar_textos(noticias)
cenarios, noticias_elegiveis = construir_cenarios(mapeadas, principal, marcacao)

# A média dos módulos usa somente as notícias retidas em cada cenário.
for nome, base_cenario in list(cenarios.items()):
    textos_retidos = noticias_elegiveis
    if 'Sem retrospectivas/ambíguas' in nome:
        textos_retidos = textos_retidos.loc[textos_retidos['marcacao'].eq('sem_indicio_explicito')]
    elif 'Sem retrospectivas' in nome:
        textos_retidos = textos_retidos.loc[textos_retidos['marcacao'].ne('retrospectiva_explicita')]
    cenarios[nome] = anexar_intensidades(base_cenario, textos_retidos)

### Etapa 2 — Calcular os resumos

Contamos os pares retidos e calculamos as correlações. Os laços percorrem, nesta ordem, cenário, empresa e métrica.


In [12]:
restrita = cenarios['Mínimo 3']
retencao = principal.groupby('stock').size().rename('principal').to_frame()
retencao['minimo_3'] = restrita.groupby('stock').size().reindex(retencao.index, fill_value=0)
retencao['retencao_pct'] = 100 * retencao['minimo_3'] / retencao['principal']
resumo_cenarios = pd.DataFrame([
    {'cenario': nome, 'pregoes': len(df), 'noticias': int(df['n_noticias'].sum()),
     'retencao_pct': 100 * len(df) / len(principal)} for nome, df in cenarios.items()])
display(resumo_cenarios.loc[resumo_cenarios['cenario'].isin(CENARIOS)].round(1))
retencao_cenarios = resumir_retencao(cenarios)
metricas = [('escore_sentimento', 'retorno_pct', 'Escore × retorno'),
            ('intensidade_liquida', 'retorno_absoluto_pct', 'Intensidade líquida × |retorno|'),
            ('n_noticias', 'retorno_absoluto_pct', 'Quantidade × |retorno|'),
            ('intensidade_media', 'retorno_absoluto_pct', 'Intensidade média × |retorno|')]
resultados = []
for nome, dados in cenarios.items():
    for empresa, grupo in dados.groupby('stock'):
        for x, y, metrica in metricas:
            resultados.append({
                'cenario': nome,
                'stock': empresa,
                'metrica': metrica,
                'n': len(grupo),
                'rho': spearman(grupo, x, y),
            })
sensibilidade_completa = pd.DataFrame(resultados)
sensibilidade = sensibilidade_completa.loc[sensibilidade_completa['cenario'].isin(CENARIOS)].copy()

classes_sensibilidade = pd.concat([resumir_classes(df).assign(cenario=nome)
    for nome, df in cenarios.items()], ignore_index=True)

,cenario,pregoes,noticias,retencao_pct
0,Principal,2387,6542,100.0
1,Mínimo 3,783,4384,32.8
2,Sem retrospectivas,2271,5921,95.1
3,Sem retrospectivas + mínimo 3,697,3807,29.2


### Etapa 3 — Desenhar e comparar

O gráfico usa os coeficientes calculados acima; a tabela permite consultar seus valores. Os nomes abreviados da legenda se referem às retrospectivas identificadas pela regra.


### Retenção e sensibilidade por empresa

As comparações usam a amostra principal de cada empresa como referência.

In [13]:
# Retenção e coeficientes por empresa e cenário, com referência na amostra principal.
referencia = (sensibilidade_completa.loc[sensibilidade_completa['cenario'].eq('Principal'),
    ['stock', 'metrica', 'n', 'rho']]
    .rename(columns={'n': 'n_principal', 'rho': 'rho_principal'}))
comparacao_robustez = sensibilidade_completa.merge(referencia,
    on=['stock', 'metrica'], validate='many_to_one')
comparacao_robustez['retencao_pct'] = 100 * comparacao_robustez['n'] / comparacao_robustez['n_principal']
comparacao_robustez['delta_rho'] = comparacao_robustez['rho'] - comparacao_robustez['rho_principal']
comparacao_robustez['sinal'] = [
    'Indefinido' if pd.isna(r) or pd.isna(ref) else
    'Coeficiente zero' if r == 0 or ref == 0 else
    'Mesmo sinal' if (r > 0) == (ref > 0) else 'Sinal invertido'
    for r, ref in zip(comparacao_robustez['rho'], comparacao_robustez['rho_principal'])]
retencao_empresa = (comparacao_robustez.loc[comparacao_robustez['cenario'].isin(CENARIOS),
    ['stock', 'cenario', 'n_principal', 'n', 'retencao_pct']]
    .drop_duplicates().sort_values(['stock', 'cenario']))
display(retencao_empresa.round(1))
display(comparacao_robustez.loc[comparacao_robustez['cenario'].isin(CENARIOS),
    ['stock', 'cenario', 'metrica', 'n', 'retencao_pct', 'rho', 'delta_rho', 'sinal']].round(4))

,stock,cenario,n_principal,n,retencao_pct
20,BAC,Mínimo 3,438,88,20.1
0,BAC,Principal,438,438,100.0
40,BAC,Sem retrospectivas,438,409,93.4
60,BAC,Sem retrospectivas + mínimo 3,438,74,16.9
24,JNJ,Mínimo 3,446,123,27.6
4,JNJ,Principal,446,446,100.0
44,JNJ,Sem retrospectivas,446,438,98.2
64,JNJ,Sem retrospectivas + mínimo 3,446,116,26.0
28,KO,Mínimo 3,334,58,17.4
8,KO,Principal,334,334,100.0


,stock,cenario,metrica,n,retencao_pct,rho,delta_rho,sinal
0,BAC,Principal,Escore × retorno,438,100.0000,0.0150,0.0000,Mesmo sinal
1,BAC,Principal,Intensidade líquida × |retorno|,438,100.0000,0.0112,0.0000,Mesmo sinal
2,BAC,Principal,Quantidade × |retorno|,438,100.0000,0.0327,0.0000,Mesmo sinal
3,BAC,Principal,Intensidade média × |retorno|,438,100.0000,0.0211,0.0000,Mesmo sinal
4,JNJ,Principal,Escore × retorno,446,100.0000,0.0307,0.0000,Mesmo sinal
...,...,...,...,...,...,...,...,...
75,NFLX,Sem retrospectivas + mínimo 3,Intensidade média × |retorno|,286,45.6140,-0.0329,-0.1057,Sinal invertido
76,NVDA,Sem retrospectivas + mínimo 3,Escore × retorno,166,30.6273,0.0123,0.0530,Sinal invertido
77,NVDA,Sem retrospectivas + mínimo 3,Intensidade líquida × |retorno|,166,30.6273,-0.0883,-0.1043,Sinal invertido
78,NVDA,Sem retrospectivas + mínimo 3,Quantidade × |retorno|,166,30.6273,0.1306,0.0054,Mesmo sinal


In [14]:
q7 = alt.Chart(sensibilidade).mark_point(filled=True, size=65).encode(
    x=alt.X('rho:Q', title='Spearman (ρ)', scale=alt.Scale(domain=[-1, 1])),
    y=alt.Y('stock:N', title='Empresa', sort=EMPRESAS),
    yOffset=alt.YOffset('cenario:N', sort=CENARIOS),
    color=alt.Color('cenario:N', title='Cenário', scale=alt.Scale(domain=CENARIOS,
        range=['#32648D', '#BD5B3A', '#16877B', '#8664A5']),
        legend=alt.Legend(orient='bottom', columns=2, labelLimit=300)),
    shape=alt.Shape('cenario:N', title='Cenário', scale=alt.Scale(domain=CENARIOS,
        range=['circle', 'square', 'triangle-up', 'cross']),
        legend=alt.Legend(orient='bottom', columns=2, labelLimit=300)),
    tooltip=['stock:N', 'cenario:N', 'n:Q', alt.Tooltip('rho:Q', format='.4f')]
).properties(width=350, height=260).facet(
    facet=alt.Facet('metrica:N', title=None, sort=[m[2] for m in metricas]), columns=2
).properties(title=alt.TitleParams(
    text='7 · Os filtros mudam a amostra e algumas associações',
    subtitle='A exclusão de retrospectivas alcança somente as identificadas pela regra.'))
display(q7)
display(sensibilidade.pivot(index=['stock', 'cenario'], columns='metrica', values='rho').round(3))

alt.FacetChart(...)

metrica                              Escore × retorno  ...  Quantidade × |retorno|
stock cenario                                          ...                        
BAC   Mínimo 3                                  0.093  ...                  -0.159
      Principal                                 0.015  ...                   0.033
      Sem retrospectivas                        0.004  ...                  -0.010
      Sem retrospectivas + mínimo 3            -0.058  ...                  -0.094
JNJ   Mínimo 3                                 -0.095  ...                   0.011
      Principal                                 0.031  ...                   0.030
      Sem retrospectivas                        0.055  ...                   0.037
      Sem retrospectivas + mínimo 3            -0.110  ...                   0.078
KO    Mínimo 3                                 -0.196  ...                   0.287
      Principal                                 0.002  ...                   0.161
      Sem retrospectivas                        0.008  ...                   0.151
      Sem retrospectivas + mínimo 3            -0.235  ...                   0.355
NFLX  Mínimo 3                                  0.051  ...                   0.103
      Principal                                 0.058  ...                   0.057
      Sem retrospectivas                        0.042  ...                   0.057
      Sem retrospectivas + mínimo 3             0.035  ...                   0.136
NVDA  Mínimo 3                                 -0.012  ...                   0.123
      Principal                                -0.041  ...                   0.125
      Sem retrospectivas                       -0.009  ...                   0.146
      Sem retrospectivas + mínimo 3             0.012  ...                   0.131

[20 rows x 4 columns]

**Resultado.** A exclusão das retrospectivas identificadas pela regra retira 621 das 6.542 associações presentes nas janelas principais. Restam 2.271 pares (95,1%), e o filtro combinado mantém 697 (29,2%). Exigir três notícias sozinho mantém 783. Entre os pares que sobrevivem à exclusão, 359 escores e 113 classes mudam após a reagregação; portanto, somente retirar pregões inteiros não reproduziria esse cenário.

Em KO, escore × retorno passa de +0,002 na principal para +0,008 sem retrospectivas e −0,235 no combinado (55 pares). Em BAC, vai de +0,015 para +0,004 e −0,058; em NFLX, de +0,058 para +0,042 e +0,035. As intensidades também não seguem uma direção comum. Assim, a exclusão isolada mantém a leitura de associações pequenas, enquanto a restrição de quantidade e sua combinação alteram a composição e alguns sinais.

**Sensibilidade da marcação.** Retirar também os sinais não conclusivos deixa 2.243 pares, ou 670 com mínimo de três. Esse filtro ampliado pode remover previsões ou notícias operacionais; serve para explorar a incerteza da regra, não para identificar todas as retrospectivas. As tabelas registram os seis cenários e a retenção por empresa/ano.

**Resposta.** Não encontramos um padrão uniforme que permaneça entre empresas e filtros. A estabilidade de uma conclusão ampla de associação fraca não equivale à estabilidade de cada coeficiente. Seleção da amostra, grupos pequenos e erros da marcação limitam a robustez; os resultados não demonstram causalidade ou capacidade preditiva.

A tabela por empresa explicita a mudança de composição: `n_principal`, `n` retido e retenção percentual. As duas intensidades usam os textos que restam em cada cenário. `delta_rho` e mudança de sinal são descrições da sensibilidade, não testes de diferença entre coeficientes.

## 11. Conclusões e limitações

A cobertura é desigual (H1), neutro é a classe mais frequente e as distribuições de retorno por classe se sobrepõem. A intensidade líquida não acompanha claramente maiores oscilações na amostra principal (H2). Há mudanças de sinal entre empresas e anos (H3), com correlações pequenas. Os quatro cenários de robustez mantêm a ausência de um padrão uniforme; exigir três notícias, isoladamente ou após excluir retrospectivas, altera a composição e alguns sinais.

A inspeção dos extremos mostra que uma manchete isolada pode produzir escore muito alto ou baixo. Exigir três reduz alguns extremos, mas conserva textos sobre outras empresas e o mercado. Quantidade de registros não garante relevância, diversidade de fontes ou precisão do sentimento para cada ticker.

As conclusões se limitam a cinco empresas, três anos e uma fonte, com horários pouco informativos, classificação do texto inteiro, vínculos indiretos de ticker e dependência entre observações. A regra de retrospectivas é conservadora, automática, pode deixar passar relatos explícitos e não foi validada por anotação humana independente. O estudo não controla fatores de mercado nem estima efeitos causais ou desempenho preditivo.

| Hipótese | Resultado neste recorte | Limite |
| --- | --- | --- |
| H1 — cobertura desigual | Apoio descritivo | Descreve o corpus, não toda a cobertura do mercado. |
| H2 — intensidade e oscilação | Sem apoio uniforme nas duas definições | Não exclui outras medidas ou relações não monotônicas. |
| H3 — empresas e anos | Heterogeneidade descritiva | Variação dos coeficientes não demonstra diferenças estruturais. |

As melhorias de intensidade e robustez foram incorporadas nas próprias células. A validação humana independente das retrospectivas e dos vínculos de ticker permanece uma limitação. A revisão humana anterior de sentimento foi preservada; não houve nova anotação humana nesta rodada.

## 12. Reprodutibilidade e organização da entrega

Este notebook é a fonte principal do artigo. As transformações, medidas e visualizações Altair são construídas em células executáveis. Os snapshots e as probabilidades incluídos permitem refazer a análise sem treinar o FinBERT ou baixar novos preços.

Execute as células em ordem e salve. A última célula exporta tabelas para `docs/analise_final/`. A regra de retrospectivas permanece documentada em `docs/retrospectivas/`. Para exportar a leitura, rode `python tools/exportar_artigo.py`; esse comando lê o notebook e não substitui sua estrutura por um site ou um gerador de células.

O retorno do professor é atendido pelos tooltips da Q3 com manchetes reais. A conferência visual dos títulos e dos seletores deve ser feita no Jupyter ou no HTML. Apresentação de dez minutos, contribuições efetivas e envio à disciplina são tarefas separadas do grupo.

In [15]:
TAB = RAIZ / 'docs/analise_final'
TAB.mkdir(parents=True, exist_ok=True)
amostras = painel.groupby('stock').agg(painel=('trade_date', 'size'), com_noticias=('tem_noticias', 'sum'))
amostras = amostras.join(retencao)
amostras['sem_noticias'] = amostras['painel'] - amostras['com_noticias']
amostras['com_noticias_janela_incompleta'] = amostras['com_noticias'] - amostras['principal']
cobertura_mensal = mensal.assign(mes=mensal['mes'].dt.strftime('%Y-%m')).pivot(index='stock', columns='mes', values='n')
tabelas = {'amostras': amostras.reset_index(), 'q1_cobertura_mensal': cobertura_mensal.reset_index(),
    'q2_classes': classes.reset_index(), 'q2_proporcoes': (100 * classes.div(classes.sum(axis=1), axis=0)).reset_index(),
    'q3_diario': diario.rename(columns={'escore_sentimento': 'escore'}),
    'q3_extremos': extremos.rename(columns={'escore_sentimento': 'escore'}),
    'q4_retornos_classes': retornos_classes, 'q5_associacoes': associacoes_q5,
    'q6_correlacoes': correlacoes.drop(columns='rotulo'), 'q7_sensibilidade': sensibilidade,
    'q7_retornos_classes': classes_sensibilidade,
    'q7_resumo_cenarios': resumo_cenarios, 'q7_retencao_empresa_ano': retencao_cenarios,
    'q7_sensibilidade_completa': sensibilidade_completa,
    'q7_comparacao': comparacao_robustez, 'q7_retencao_empresa': retencao_empresa,
    'q3_extremos_com_suporte': extremos_suporte.rename(columns={'escore_sentimento': 'escore'})}
for nome, tabela in tabelas.items():
    tabela.to_csv(TAB / f'{nome}.csv', index=False, float_format='%.10g', date_format='%Y-%m-%d')
textos_extremos = noticias.merge(extremos[['stock', 'news_date', 'extremo']],
    on=['stock', 'news_date'], validate='many_to_one').sort_values(['stock', 'news_date', 'source_row'])
textos_extremos.to_csv(DATA / 'manchetes_extremos_analise.csv', index=False, date_format='%Y-%m-%d')
print(f'{len(tabelas)} tabelas exportadas; {len(textos_extremos)} manchetes nos extremos.')
textos_suporte = noticias.merge(extremos_suporte[['stock', 'news_date', 'extremo']],
    on=['stock', 'news_date'], validate='many_to_one')
textos_suporte.to_csv(TAB / 'q3_manchetes_extremos_com_suporte.csv', index=False, date_format='%Y-%m-%d')
marcacao.to_csv(RAIZ / 'docs/retrospectivas/marcacao_textos.csv', index=False)
pd.concat([df.assign(cenario=nome) for nome, df in cenarios.items()], ignore_index=True).to_csv(
    DATA / 'base_robustez_cenarios.csv', index=False, date_format='%Y-%m-%d', float_format='%.12g')

principal[['stock', 'trade_date', 'intensidade_liquida', 'intensidade_media']].to_csv(
    DATA / 'medidas_intensidade_final.csv', index=False, date_format='%Y-%m-%d')

17 tabelas exportadas; 10 manchetes nos extremos.


## Referências

[1] AENLLE, Miguel. *Daily Financial News for 6000+ Stocks*. Kaggle. [Base de notícias](https://www.kaggle.com/datasets/miguelaenlle/massive-stock-news-analysis-db-for-nlpbacktests). Arquivo utilizado: `raw_analyst_ratings.csv`.

[2] YAHOO FINANCE; YFINANCE. *Cotações históricas e documentação de yfinance.download*. [Documentação de coleta](https://ranaroussi.github.io/yfinance/reference/api/yfinance.download.html). Snapshot utilizado: 09/10/2026; parâmetros e hashes no [registro de integração](docs/proveniencia.json).

[3] PROSUSAI. *FinBERT*. Hugging Face. [Modelo e documentação](https://huggingface.co/ProsusAI/finbert); [revisão utilizada](https://huggingface.co/ProsusAI/finbert/tree/4556d13015211d73dccd3fdd39d39232506f3e43).

[4] ARACI, Dogu. *FinBERT: Financial Sentiment Analysis with Pre-trained Language Models*. 2019. [arXiv:1908.10063](https://arxiv.org/abs/1908.10063).

[5] EXCHANGE_CALENDARS; ICE/NYSE. Calendário XNYS e comunicados históricos. [Biblioteca](https://github.com/gerrymanoim/exchange_calendars); [calendário 2017–2019](https://ir.theice.com/press/news-details/2016/NYSE-Group-announces-2018-and-2019-holiday-and-early-closings-calendar/default.aspx); [calendário 2018–2020](https://ir.theice.com/press/news-details/2017/NYSE-Group-Announces-2018-2019-and-2020-Holiday-and-Early-Closings-Calendar/default.aspx); [fechamento extraordinário de 05/12/2018](https://ir.theice.com/press/news-details/2018/New-York-Stock-Exchange-to-Honor-President-George-H-W-Bush/default.aspx).

[6] RULFF, João. *Projeto Final — Análise Exploratória de Dados e Visualização*. FGV EMAp, 2026.2, p. 1–2. Enunciado fornecido à turma, arquivo `AED20262_projeto.pdf`. A proposta e o cronograma do grupo são documentos internos descritos no [índice documental](docs/README.md).

[7] PANDAS; VEGA-ALTAIR. Documentação de [correlação](https://pandas.pydata.org/docs/reference/api/pandas.Series.corr.html), [postos](https://pandas.pydata.org/docs/reference/api/pandas.Series.rank.html) e [boxplots](https://altair-viz.github.io/user_guide/marks/boxplot.html). Consultada em 16/10/2026.

[8] VEGA-ALTAIR. [Interações](https://altair-viz.github.io/user_guide/interactions/parameters.html) e [exportação de gráficos](https://altair-viz.github.io/user_guide/saving_charts.html). Consultadas em 16/10/2026.
